In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [2]:
df = pd.read_pickle('cleaned_chicago_crime_data.pkl')

In [3]:
pd.set_option('display.max_columns', None)

In [4]:
# Drop unnecessary columns for clustering
df = df.drop(columns=['Case Number','Location', 'ID', 'Updated On', 'Block', 'Description', 'IUCR', 'FBI Code'])

In [5]:
# Since X co-oedinate is highly co-related to Longitude, we romove X-coordinate column. 
# Also, Y co-oedinate is highly co-related to Latitude, we romove Y-coordinate column. 
df = df.drop(columns=['X Coordinate', 'Y Coordinate'])

In [6]:
# # Create temporal features
df["Month"] = df["Date"].dt.month
df["Day_of_Week"] = df["Date"].dt.day_name()
df["Hour"] = df["Date"].dt.hour
df["Is_Weekend"] = df["Day_of_Week"].isin(["Saturday", "Sunday"]).astype(int)

def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Fall"

df["Season"] = df["Month"].apply(get_season)

In [7]:
# geographical features
# district alreay in the data

# coordinate binning
df["Lat_bin"] = pd.cut(df["Latitude"], bins=20)
df["Lon_bin"] = pd.cut(df["Longitude"], bins=20)

In [8]:
# creating crime severity column
severity_map = {
    "HOMICIDE": 5,
    "CRIMINAL SEXUAL ASSAULT": 5,
    "KIDNAPPING": 5,

    "ROBBERY": 4,
    "WEAPONS VIOLATION": 4,
    "ARSON": 4,
    "HUMAN TRAFFICKING": 4,

    "ASSAULT": 3,
    "BATTERY": 3,
    "SEX OFFENSE": 3,
    "STALKING": 3,
    "INTIMIDATION": 3,
    "OFFENSE INVOLVING CHILDREN": 3,

    "BURGLARY": 2,
    "MOTOR VEHICLE THEFT": 2,
    "CRIMINAL DAMAGE": 2,
    "CRIMINAL TRESPASS": 2,
    "DECEPTIVE PRACTICE": 2,

    "THEFT": 1,
    "NARCOTICS": 1,
    "PROSTITUTION": 1,
    "PUBLIC PEACE VIOLATION": 1,
    "LIQUOR LAW VIOLATION": 1,
    "GAMBLING": 1,
    "OBSCENITY": 1,
    "NON-CRIMINAL": 1,
    "OTHER OFFENSE": 1,
    "OTHER NARCOTIC VIOLATION": 1,
    "INTERFERENCE WITH PUBLIC OFFICER": 1,
    "PUBLIC INDECENCY": 1,
    "CONCEALED CARRY LICENSE VIOLATION": 1
}

df["Crime_Severity_Score"] = df["Primary Type"].map(severity_map)

In [9]:
# Categorical Encoding for Crime type and Location Description

# Frequency encoding for Location Description and crime type
freq = df["Location Description"].value_counts(normalize=True)
df["Location_Desc_freq"] = df["Location Description"].map(freq)

freq = df["Primary Type"].value_counts(normalize=True)
df["Primary Type_freq"] = df["Primary Type"].map(freq)

In [10]:
# Normalizing location coordinates
scaler = StandardScaler()

df[["Latitude_Norm", "Longitude_Norm"]] = scaler.fit_transform(
    df[["Latitude", "Longitude"]]
)

In [11]:
df['Arrest'] = df['Arrest'].astype(int)
df['Domestic'] = df['Domestic'].astype(int)

In [12]:
encoder = LabelEncoder()
df['Primary Type'] = encoder.fit_transform(df['Primary Type'])
df['Location Description'] = encoder.fit_transform(df['Location Description'])
df['Day_of_Week'] = df['Day_of_Week'].replace({'Monday':0, 'Tuesday':1, 'Wednesday':2, 'Thursday':'3', 'Friday':4, 'Saturday':5, 'Sunday':6}).astype(int)
df['Season'] = encoder.fit_transform(df['Season'])

In [13]:
df.head()

,Date,Primary Type,Location Description,Arrest,Domestic,Beat,Ward,Year,Latitude,Longitude,District,Community Area,Month,Day_of_Week,Hour,Is_Weekend,Season,Lat_bin,Lon_bin,Crime_Severity_Score,Location_Desc_freq,Primary Type_freq,Latitude_Norm,Longitude_Norm
0,2026-03-18,2,16,0,1,332,5,2026,41.768797,-87.584413,3,43,3,2,0,0,1,"(41.758, 41.777]","(-87.586, -87.566]",3,0.194385,0.180824,-0.898721,1.418079
1,2026-03-18,29,130,0,0,411,8,2026,41.745255,-87.584152,4,45,3,2,0,0,1,"(41.739, 41.758]","(-87.586, -87.566]",1,0.015273,0.234081,-1.171113,1.422509
2,2026-03-18,29,103,0,0,1732,30,2026,41.943437,-87.721197,17,21,3,2,0,0,1,"(41.928, 41.947]","(-87.73, -87.709]",1,0.119159,0.234081,1.121917,-0.900134
4,2026-03-18,8,103,0,0,735,17,2026,41.762156,-87.673463,7,67,3,2,0,0,1,"(41.758, 41.777]","(-87.689, -87.668]",2,0.119159,0.057381,-0.975564,-0.091136
5,2026-03-18,3,119,0,0,1733,33,2026,41.940161,-87.701524,17,21,3,2,0,0,1,"(41.928, 41.947]","(-87.709, -87.689]",2,0.267355,0.038438,1.084011,-0.566714


In [14]:
df.shape

(495708, 24)

In [15]:
df.to_pickle('engineered_chicago_crime_data.pkl')